# 🛋️ AI Interior Designer — سرور مدل روی Colab

این نوت‌بوک، بک‌اند هوش مصنوعی (**SDXL** + ControlNet + Inpainting + YOLOv8 + SAM) پروژهٔ **InteriorAI** را روی GPU رایگان گوگل کولب اجرا می‌کند و از طریق یک تونل امن به بک‌اند محلی پروژه (Flask روی سیستم خودت) وصل می‌شود.

## ✅ قبل از شروع

| مرحله | کار |
|---|---|
| ۱ | بالا سمت راست → **Runtime → Change runtime type → T4 GPU** را انتخاب و ذخیره کن |
| ۲ | یک توکن رایگان از [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken) بگیر (ثبت‌نام رایگان، چند ثانیه) |
| ۳ | روی سیستم خودت، بک‌اند محلی پروژه را اجرا کن: فایل `START_ORIGINAL_BACKEND.cmd` |
| ۴ | از منوی بالا: **Runtime → Run all** را بزن و صبر کن |

## ⏱️ زمان تقریبی
بار اول حدود ۱۰ تا ۱۵ دقیقه طول می‌کشد (دانلود مدل SDXL، حدود ۱۷ گیگابایت مجموع). دفعات بعد در همین سشن سریع‌تر است.

## 🔑 در پایان اجرا
آخرین سلول یک بار authtoken ngrok را می‌پرسد (مخفی وارد کن) و دو خط چاپ می‌کند:

```
BACKEND_URL: https://xxxx-xxxx.ngrok-free.app
CONNECTION_KEY: ...........................
```

این دو مقدار را در فرانت، داخل پنجرهٔ **«Connect AI Backend» → حالت Google Colab** بچسبان و Connect بزن. از این لحظه، تولید عکس واقعی و رایگان است — تا وقتی این تب Colab باز و روشن بماند.

> ⚠️ اگر تب Colab را ببندی یا رانتایم قطع شود، اتصال قطع می‌شود و باید دوباره از اول (Runtime → Run all) اجرا کنی.


## ۱) بررسی GPU
مطمئن شو Runtime روی T4 GPU تنظیم شده — اگر خطا داد، از منو Runtime → Change runtime type را عوض کن و این سلول را دوباره اجرا کن.

In [ ]:
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU first."
print(torch.cuda.get_device_name(0))


## ۲) نصب کتابخانه‌ها
حدود ۱ تا ۲ دقیقه طول می‌کشد.

In [ ]:
import subprocess, sys, os
os.environ["USE_TF"]="0"
subprocess.check_call([sys.executable,"-m","pip","install","-q","diffusers==0.37.0","transformers==4.57.6","accelerate==1.12.0","flask","flask-cors","pyngrok","ultralytics","segment-anything","opencv-python-headless","safetensors"])


## ۳) آماده‌سازی مسیرهای ذخیره‌سازی

In [ ]:
import os, torch
CACHE_DIR="/content/interior_original/models"
OUTPUT_DIR="/content/interior_original/outputs"
os.makedirs(CACHE_DIR,exist_ok=True)
os.makedirs(OUTPUT_DIR,exist_ok=True)


## ۴) بارگذاری مدل‌ها
پنج سلول بعدی مدل‌های ControlNet (SDXL)، Stable Diffusion XL، Inpainting (SDXL)، YOLOv8 و SAM را دانلود و روی GPU بارگذاری می‌کنند — بار اول به‌خاطر حجم بالاتر SDXL چند دقیقه بیشتر طول می‌کشد، بعدش کش می‌شود.

In [ ]:
from diffusers import ControlNetModel, AutoencoderKL
import torch

print("Loading SDXL ControlNet (Canny)... (bigger than SD1.5, first time is slower)")

controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0",
    torch_dtype=torch.float16,
    cache_dir=CACHE_DIR
)

# SDXL's default VAE produces NaNs in fp16 — this community fix is the
# standard fix everyone uses, recommended directly on the SDXL model card.
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16,
    cache_dir=CACHE_DIR
)

print("✅ SDXL ControlNet + VAE loaded!")

In [ ]:
from diffusers import StableDiffusionXLControlNetPipeline, UniPCMultistepScheduler

print("Loading SDXL... (a real quality upgrade from SD1.5 — first load downloads ~7GB, be patient)")

style_pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    cache_dir=CACHE_DIR
)
style_pipe.scheduler = UniPCMultistepScheduler.from_config(style_pipe.scheduler.config)
print("✅ SDXL loaded!")

# IMPORTANT: load_ip_adapter() MUST run before enable_sequential_cpu_offload().
# Doing it in the other order (as an earlier version of this notebook did)
# leaves the IP-Adapter's new image_encoder/attention layers outside the
# offload hook chain that was already wired up, causing:
#   "RuntimeError: Tensor on device cuda:0 is not on the expected device meta!"
print("Loading IP-Adapter for SDXL... (lets you insert a specific item from a reference photo)")
style_pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
style_pipe.set_ip_adapter_scale(0.0)  # neutral by default; add_object_from_reference raises this when needed
print("✅ IP-Adapter loaded!")

# Offloading goes LAST, after every component (including IP-Adapter) exists.
style_pipe.enable_sequential_cpu_offload()
style_pipe.enable_vae_tiling()
print("✅ Offloading enabled!")

In [ ]:
from diffusers import AutoPipelineForInpainting

print("Loading SDXL Inpainting...")

inpaint_pipe = AutoPipelineForInpainting.from_pretrained(
    "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
    torch_dtype=torch.float16,
    cache_dir=CACHE_DIR
)
# See the note above the SDXL cell — sequential offload trades speed for
# staying safely under T4's 16GB VRAM.
inpaint_pipe.enable_sequential_cpu_offload()

print("✅ SDXL Inpainting loaded!")

In [ ]:
from ultralytics import YOLO

print("Loading YOLOv8...")

# Kept on CPU (like SAM below) so it doesn't compete with SDXL/ControlNet for
# GPU memory — object detection is fast enough on CPU that this is worth it
# for the extra VRAM headroom on a free T4.
yolo_model = YOLO("yolov8x.pt")
yolo_model.to("cpu")

print("✅ YOLOv8 loaded!")

In [ ]:
from segment_anything import sam_model_registry, SamPredictor
import requests
import os

sam_path = "/content/sam_vit_h.pth"

if not os.path.exists(sam_path):
    print("Downloading SAM model... (2-3 minutes)")
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(sam_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print("✅ SAM downloaded!")
else:
    print("✅ SAM already exists, skipping download!")

sam = sam_model_registry["vit_h"](checkpoint=sam_path)
sam.to("cpu")
sam_predictor = SamPredictor(sam)

print("✅ SAM loaded and on GPU!")

## ۵) توابع کمکی (تبدیل تصویر، لبه‌یابی Canny)

In [ ]:
import base64
import io
import numpy as np
import cv2
from PIL import Image

def base64_to_pil(b64_string):
    # Frontend sometimes sends a full data URL (from FileReader.readAsDataURL);
    # base64.b64decode() silently mangles "data:image/...;base64," into garbage
    # bytes instead of raising, so strip it explicitly.
    if b64_string.startswith("data:") and "," in b64_string:
        b64_string = b64_string.split(",", 1)[1]
    img_data = base64.b64decode(b64_string)
    return Image.open(io.BytesIO(img_data)).convert("RGB").resize((768, 768))

def pil_to_base64(pil_image):
    buffer = io.BytesIO()
    pil_image.save(buffer, format="JPEG", quality=95)
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

def get_canny_edges(pil_image):
    img_np = np.array(pil_image)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    edges_rgb = cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(edges_rgb)

print("✅ Helper functions ready! (base64_to_pil now strips data-URL prefixes)")

## ۶) پرامپت‌های ۸ سبک طراحی

In [ ]:
STYLE_PROMPTS = {
    "minimalist": {
        "prompt": "minimalist interior design, pure white and warm beige walls, polished concrete or light wood floor, clean geometric lines, no clutter, abundant natural daylight, recessed ceiling lights, hidden storage, Scandinavian and Japanese wabi-sabi influence, breathing space, monochromatic white and beige palette, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "cluttered, colorful, busy, ornate, dark, multiple patterns, too many objects, cheap, dirty, low quality, blurry, grainy, distorted, watermark, ugly, oversaturated"
    },
    "industrial": {
        "prompt": "industrial interior design, raw exposed red brick walls, polished concrete floor, black steel window frames, exposed metal ceiling beams, Edison bulb pendant lights, vintage leather accents, reclaimed dark wood and iron furniture, urban warehouse aesthetic, raw metal pipes visible, matte black hardware, leather and metal textures, moody warm lighting, Chicago loft style, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "soft, pastel, floral, traditional, plastic, cheap, colorful, bright white, polished, fancy, ornate, blurry, low quality, grainy, watermark"
    },
    "cyberpunk": {
        "prompt": "cyberpunk interior design, dark charcoal walls with hexagonal panels, RGB LED strip lighting glowing blue and purple, neon pink and cyan signs on wall, holographic display panels, carbon fiber textures on furniture, metallic chrome accents, city skyline visible through large window at night with rain, sleek dark furniture with glowing edges, tech gadgets, futuristic floating shelves, neon reflections on floor, Blade Runner inspired, cinematic lighting, ultra sharp, 8k resolution, photorealistic",
        "negative": "foggy, hazy, too dark, obscured furniture, blurry, low quality, traditional, wooden, natural, daytime, bright white, plain walls, no tech elements, watermark, grainy"
    },
    "modern_luxury": {
        "prompt": "ultra luxury modern interior design, Calacatta marble accent wall with gold veining, herringbone light oak hardwood floor, upholstered furniture in cream boucle fabric, sculptural gold brass chandelier, floor to ceiling silk curtains in ivory, marble surfaces with gold legs, designer artwork in gold frames, cashmere throw accents, fresh white orchids in crystal vase, hidden ambient lighting in ceiling coves, five star hotel suite quality, Versace and Fendi inspired, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "cheap, basic, plastic, clutter, industrial, rustic, dark, crowded, low budget, synthetic materials, blurry, grainy, low quality, watermark, distorted"
    },
    "scandinavian": {
        "prompt": "Scandinavian hygge interior design, white washed pine plank floors, pale sage green accent wall, natural oak surfaces, soft wool throws in oatmeal color, rattan pendant light, potted fiddle leaf fig plant, sheepskin rug, floating oak shelves with ceramic vases, linen curtains filtering soft morning light, dried pampas grass in terracotta pot, candles on windowsill, simple geometric cushions in muted tones, Copenhagen apartment style, cozy and warm atmosphere, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "dark, ornate, cluttered, colorful, heavy patterns, gilded, industrial, cold, sterile, cheap, plastic, blurry, low quality, grainy, watermark"
    },
    "midcentury_modern": {
        "prompt": "mid century modern interior design, warm walnut teak wood furniture, geometric patterned wool rug in orange and brown, Eames style lounge chair, tulip side table, sunburst wall clock in gold, abstract 1960s artwork, warm Edison bulb floor lamp with tripod legs, avocado green accent wall, tapered furniture legs, atomic age decorative objects, teak credenza, retro record player on shelf, warm amber lighting, Mad Men inspired, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "contemporary, futuristic, traditional, ornate, dark, gothic, cold colors, cheap, plastic, modern minimalist, blurry, low quality, grainy, watermark"
    },
    "japanese_zen": {
        "prompt": "Japanese zen interior design, natural tatami-textured flooring, shoji screen panels with warm backlight, natural unfinished hinoki wood walls, bamboo ceiling accents, bonsai tree on wooden stand, smooth river stones arrangement, ikebana flower arrangement in ceramic vase, washi paper pendant lamp, moss garden view through window, earthy tones of sand beige and forest green, negative space philosophy, wabi-sabi imperfection, Kyoto ryokan inspired, ultra sharp, 8k resolution, photorealistic, professional interior photography",
        "negative": "cluttered, colorful, western, modern tech, busy patterns, gold, ornate, plastic, synthetic, noisy, loud, western furniture, blurry, low quality, grainy, watermark"
    },
    "bohemian": {
        "prompt": "bohemian interior design, terracotta painted walls, layered Persian and Moroccan rugs on wooden floor, macrame wall hanging, hanging rattan egg chair accent, collection of trailing plants in ceramic and woven pots, warm string fairy lights, gallery wall of eclectic vintage art, colorful embroidered cushions stacked high, suzani throw accents, beaded curtains, incense holder on vintage surface, warm golden hour lighting, Marrakech riad inspired, professional interior photography, ultra sharp, 8k resolution, photorealistic, architectural digest style",
        "negative": "minimal, plain, cold, sterile, corporate, modern sleek, industrial, white walls, no plants, sparse, empty, blurry, low quality, grainy, watermark"
    }
}

# Applied to every style so the room's actual function is never changed
# (e.g. a kitchen must stay a kitchen — earlier prompts said "bedroom" for
# every style, which is why kitchens/living rooms were turning into bedrooms).
ROOM_PRESERVE_SUFFIX = ", keep the exact same room type and function, same walls, same windows, same doors, same fixed furniture layout — only change wall and floor materials, colors, lighting fixtures and decor"
ROOM_PRESERVE_NEGATIVE = ", different room type, converted room, added bed, added bedroom furniture, structural changes, moved walls, moved windows, moved doors"

print("✅ Style prompts updated — room type/function is now preserved for every style!")
print("Available styles:", list(STYLE_PROMPTS.keys()))

## ۷) توابع اصلی تولید تصویر
سبک‌دهی، تشخیص آبجکت، ویرایش آبجکت، پیش‌نمایش همه سبک‌ها، مبله‌کردن اتاق.

In [ ]:
# Once IP-Adapter is loaded onto style_pipe, diffusers requires EVERY call to
# that pipe to receive image_embeds (via ip_adapter_image) — even calls that
# don't want it — or it raises ValueError. A neutral gray image + scale=0.0
# gives it something to encode that has zero actual influence on the output.
_IP_ADAPTER_NEUTRAL_IMAGE = Image.new("RGB", (224, 224), (128, 128, 128))


def generate_style(image_b64, style_name, palette=None, custom_prompt=None):
    print(f"Generating style: {style_name}...")

    pil_image = base64_to_pil(image_b64)
    pil_image = pil_image.resize((768, 768))
    edge_image = get_canny_edges(pil_image)

    # Use custom prompt if provided
    if custom_prompt:
        print(f"Using custom prompt: {custom_prompt}")
        base_prompt = (
            custom_prompt + ROOM_PRESERVE_SUFFIX +
            ", interior design photography, 8k, photorealistic, "
            "sharp focus, professional photography, highly detailed"
        )
        negative_prompt = (
            "blurry, low quality, low resolution, grainy, "
            "pixelated, distorted, ugly, watermark" + ROOM_PRESERVE_NEGATIVE
        )
    else:
        if style_name not in STYLE_PROMPTS:
            return {"error": f"Unknown style: {style_name}"}
        config = STYLE_PROMPTS[style_name]
        base_prompt = config["prompt"] + ROOM_PRESERVE_SUFFIX
        negative_prompt = config["negative"] + ROOM_PRESERVE_NEGATIVE
        if palette and palette.get("prompt"):
            base_prompt = (
                palette["prompt"] + ", " + base_prompt +
                ", dominant color scheme must match palette exactly"
            )
            negative_prompt += ", wrong colors, different colors"

    style_pipe.set_ip_adapter_scale(0.0)
    result = style_pipe(
        prompt=base_prompt + ", highly detailed, 8k, photorealistic, sharp focus, professional photography",
        negative_prompt=negative_prompt + ", blurry, low quality, distorted, ugly, watermark",
        image=edge_image,
        ip_adapter_image=_IP_ADAPTER_NEUTRAL_IMAGE,
        num_inference_steps=40,
        guidance_scale=9.0,
        controlnet_conditioning_scale=1.05,
        height=768,
        width=768,
    ).images[0]

    save_path = "/content/interior_original/outputs/room_styled.jpg"
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    result.save(save_path, quality=95)

    backup_path = "/content/interior_original/outputs/room_styled_original.jpg"
    result.save(backup_path, quality=95)

    print("Generation complete!")
    return {"image": pil_to_base64(result)}

print("✅ generate_style: fixed the IP-Adapter ValueError (now passes a neutral zero-influence image on regular calls)")

In [ ]:
def detect_objects(image_b64):
    print("Detecting objects...")

    drive_path = "/content/interior_original/outputs/room_styled.jpg"
    if os.path.exists(drive_path):
        pil_image = Image.open(drive_path).convert("RGB").resize((512, 512))
        print("Loaded image from Drive")
    else:
        pil_image = base64_to_pil(image_b64)
        print("Loaded image from base64")

    img_np = np.array(pil_image)
    results = yolo_model(img_np, verbose=False)

    detected = []
    for result in results:
        for box in result.boxes:
            label = yolo_model.names[int(box.cls)]
            confidence = float(box.conf)
            if confidence > 0.15:
                if label not in detected:
                    detected.append(label)

    print(f"Detected {len(detected)} objects: {detected}")
    return {"objects": detected}

print("✅ detect_objects function ready!")

In [ ]:
def edit_object(image_b64, object_label, edit_prompt):
    print(f"Editing object: {object_label}...")

    # Always load from the ORIGINAL styled image for best quality
    original_path = "/content/interior_original/outputs/room_styled_original.jpg"
    current_path = "/content/interior_original/outputs/room_styled.jpg"

    # Use original if exists, else use current
    load_path = original_path if os.path.exists(original_path) else current_path

    pil_image = Image.open(load_path).convert("RGB")
    original_size = pil_image.size
    print(f"Loaded image: {pil_image.size}")

    # Resize only for processing
    pil_resized = pil_image.resize((512, 512))
    img_np = np.array(pil_resized)

    # Step 1 - Detect objects with YOLOv8
    results = yolo_model(img_np, verbose=False)
    all_labels = []
    target_box = None

    for result in results:
        for box in result.boxes:
            label = yolo_model.names[int(box.cls)]
            confidence = float(box.conf)
            all_labels.append(f"{label} ({confidence:.2f})")
            if confidence > 0.10 and label.lower() == object_label.lower():
                target_box = box.xyxy[0].cpu().numpy()
                break

    print(f"All detected: {all_labels}")

    # Try partial match if not found
    if target_box is None:
        for result in results:
            for box in result.boxes:
                label = yolo_model.names[int(box.cls)]
                confidence = float(box.conf)
                if confidence > 0.10 and (
                    object_label.lower() in label.lower() or
                    label.lower() in object_label.lower()
                ):
                    target_box = box.xyxy[0].cpu().numpy()
                    print(f"Partial match: {label}")
                    break

    if target_box is None:
        return {"error": f"Could not find '{object_label}'. Detected: {[l.split(' ')[0] for l in all_labels]}"}

    # Step 2 - SAM mask on resized image
    sam_predictor.set_image(img_np)
    x1, y1, x2, y2 = target_box
    input_box = np.array([x1, y1, x2, y2])
    masks, _, _ = sam_predictor.predict(
        box=input_box,
        multimask_output=False
    )
    mask = masks[0]
    mask_image = Image.fromarray((mask * 255).astype(np.uint8))

    # Step 3 - Inpaint on 512x512
    full_prompt = (
        edit_prompt +
        ", highly detailed, photorealistic, 8k, sharp focus, "
        "professional interior design photography, perfect lighting, "
        "high resolution textures, architectural digest quality"
    )
    negative_prompt = (
        "low quality, blurry, distorted, ugly, deformed, "
        "pixelated, grainy, watermark, unrealistic, soft focus"
    )

    result_512 = inpaint_pipe(
        prompt=full_prompt,
        negative_prompt=negative_prompt,
        image=pil_resized,
        mask_image=mask_image,
        num_inference_steps=40,
        guidance_scale=8.5,
        strength=0.95,
    ).images[0]

    # Step 4 - Upscale back to original size
    result_final = result_512.resize(original_size, Image.LANCZOS)

    # Save edited as current — but keep original untouched
    result_final.save(current_path, quality=95)
    print(f"Saved edited image at original size: {original_size}")
    print("Object edit complete!")

    return {"image": pil_to_base64(result_final)}

print("✅ edit_object updated — no more quality loss!")

In [ ]:
def generate_all_previews(image_b64, palette=None):
    print("Generating all 8 style previews...")
    if palette:
        print(f"Using palette: {palette.get('name')}")
    previews = {}

    pil_image = base64_to_pil(image_b64)
    pil_image = pil_image.resize((512, 512))
    edge_image = get_canny_edges(pil_image)

    style_pipe.set_ip_adapter_scale(0.0)

    for style_name, config in STYLE_PROMPTS.items():
        print(f"  Generating {style_name}...")

        base_prompt = config["prompt"] + ROOM_PRESERVE_SUFFIX
        neg_prompt = config["negative"] + ROOM_PRESERVE_NEGATIVE

        if palette and palette.get("prompt"):
            base_prompt = (
                palette["prompt"] +
                ", " + base_prompt +
                ", dominant color scheme must match palette exactly"
            )
            neg_prompt += ", wrong colors, different colors, ignore palette"

        result = style_pipe(
            prompt=base_prompt,
            negative_prompt=neg_prompt,
            image=edge_image,
            ip_adapter_image=_IP_ADAPTER_NEUTRAL_IMAGE,
            num_inference_steps=25,
            guidance_scale=12.0,
            controlnet_conditioning_scale=1.0,
            height=512,
            width=512,
        ).images[0]

        previews[style_name] = pil_to_base64(result)
        print(f"  Done: {style_name}")

    print("All 8 previews complete!")
    return {"previews": previews}

print("✅ generate_all_previews: same IP-Adapter fix applied")

In [ ]:
def furnish_room(image_b64, furnish_prompt):
    print(f"Furnishing room: {furnish_prompt[:50]}...")

    pil_image = base64_to_pil(image_b64)
    pil_image = pil_image.resize((768, 768))
    edge_image = get_canny_edges(pil_image)

    full_prompt = (
        furnish_prompt +
        ", same room structure, same walls, same windows, same floor, "
        "same lighting, only add furniture, photorealistic, 8k, "
        "interior design photography, highly detailed"
    )
    negative_prompt = (
        "different room, different walls, different windows, different floor, "
        "changed structure, blurry, low quality, distorted, watermark"
    )

    result = style_pipe(
        prompt=full_prompt,
        negative_prompt=negative_prompt,
        image=edge_image,
        num_inference_steps=40,
        guidance_scale=12.0,
        controlnet_conditioning_scale=1.3,
        height=768,
        width=768,
    ).images[0]

    save_path = "/content/interior_original/outputs/room_styled.jpg"
    result.save(save_path, quality=95)
    backup_path = "/content/interior_original/outputs/room_styled_original.jpg"
    result.save(backup_path, quality=95)

    print("Furnishing complete!")
    return {"image": pil_to_base64(result)}

print("✅ furnish_room function ready!")

In [ ]:
def furnish_room_inpaint(image_b64, furnish_prompt):
    print("Furnishing room with inpainting...")

    from PIL import ImageFilter

    pil_image = base64_to_pil(image_b64)
    pil_image = pil_image.resize((512, 512))
    img_np = np.array(pil_image)
    h, w = img_np.shape[:2]

    # Wide, smooth feather + a Gaussian blur on top so the transition between
    # the preserved top and the newly-generated bottom has no visible seam —
    # the previous 8%-tall linear feather was too narrow and still left a
    # visible line where the new floor met the old one.
    mask_np = np.zeros((h, w), dtype=np.uint8)
    floor_start = int(h * 0.40)
    feather_zone = int(h * 0.30)
    mask_np[floor_start + feather_zone:, :] = 255
    for i in range(feather_zone):
        alpha = int(255 * (i / feather_zone))
        mask_np[floor_start + i, :] = alpha
    mask_image = Image.fromarray(mask_np).filter(ImageFilter.GaussianBlur(radius=6))

    full_prompt = (
        furnish_prompt +
        ", the flooring material and color continues seamlessly and matches the "
        "rest of the room exactly, consistent perspective and lighting, no visible "
        "seam, photorealistic, sharp focus, interior design, 8k, highly detailed"
    )
    negative_prompt = (
        "visible seam, patchy floor, mismatched flooring, different floor color, "
        "floating object, disconnected hardware, malformed furniture, warped metal, "
        "dark, gloomy, dim lighting, shadows, empty room, "
        "double exposure, ghosting, transparent, blurry, low quality, watermark"
    )

    print("Running inpainting...")

    result = inpaint_pipe(
        prompt=full_prompt,
        negative_prompt=negative_prompt,
        image=pil_image,
        mask_image=mask_image,
        num_inference_steps=60,
        guidance_scale=11.0,
        strength=0.85,
    ).images[0]

    result = result.resize((768, 768), Image.LANCZOS)

    save_path = "/content/interior_original/outputs/room_styled.jpg"
    result.save(save_path, quality=98)
    backup_path = "/content/interior_original/outputs/room_styled_original.jpg"
    result.save(backup_path, quality=98)

    print("Furnishing complete!")
    return {"image": pil_to_base64(result)}

print("✅ furnish_room_inpaint: wider/blurred feather zone to remove the visible floor seam, and prompt now asks explicitly for matching flooring/lighting")

In [ ]:
def add_object_from_reference(room_image_b64, object_image_b64, placement_prompt=""):
    print("Adding object from reference photo...")

    room_image = base64_to_pil(room_image_b64).resize((768, 768))
    object_image = base64_to_pil(object_image_b64)
    edge_image = get_canny_edges(room_image)

    base_prompt = (
        (placement_prompt.strip() or "place this exact item naturally in the room") +
        ", keep the exact same room type and function, same walls, same windows, "
        "same doors, same fixed furniture layout, seamlessly blended, matching "
        "perspective and lighting, photorealistic, 8k, interior design photography, highly detailed"
    )
    negative_prompt = (
        "different room, different walls, different windows, different floor, "
        "changed structure, floating object, wrong perspective, mismatched lighting, "
        "blurry, low quality, distorted, watermark"
    )

    style_pipe.set_ip_adapter_scale(0.6)
    result = style_pipe(
        prompt=base_prompt,
        negative_prompt=negative_prompt,
        image=edge_image,
        ip_adapter_image=object_image,
        num_inference_steps=40,
        guidance_scale=9.0,
        controlnet_conditioning_scale=1.0,
        height=768,
        width=768,
    ).images[0]

    save_path = "/content/interior_original/outputs/room_styled.jpg"
    result.save(save_path, quality=95)
    backup_path = "/content/interior_original/outputs/room_styled_original.jpg"
    result.save(backup_path, quality=95)

    print("Object added!")
    return {"image": pil_to_base64(result)}

print("✅ add_object_from_reference ready — send a photo of a specific item and it gets inserted into the room")

> این تابع از **IP-Adapter** استفاده می‌کنه تا شیء دقیقاً شبیه عکس مرجعی که می‌فرستی رو (مثلاً یک مبل یا آباژور خاص) داخل عکس اتاق بذاره — نه فقط یک توصیف متنی.

## ۸) سرور Flask
همان مسیرهایی (`/generate`, `/detect-objects`, ...) که بک‌اند محلی پروژه انتظارشان را دارد، به‌علاوهٔ نسخهٔ `/colab-*` برای استفادهٔ مستقیم.

In [ ]:
import os, io, base64, threading
from flask import Flask, request, jsonify
from pyngrok import ngrok
from PIL import Image

colab_app = Flask(__name__)
import secrets, hmac
CONNECTION_KEY = secrets.token_urlsafe(32)
MODEL_LOCK = threading.Lock()

UPLOAD_STORE = {"image_b64": None}
UPLOAD_FOLDER = "/content/uploads"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

# ── Manual CORS — handles preflight for ALL routes ────────────────────────────
@colab_app.after_request
def add_cors(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type, ngrok-skip-browser-warning, Authorization"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
    return response

@colab_app.before_request
def handle_options():
    if request.method != "OPTIONS" and not hmac.compare_digest(request.headers.get("Authorization", ""), "Bearer " + CONNECTION_KEY):
        return jsonify({"error":"Connection key required"}), 401
    if request.method == "OPTIONS":
        resp = colab_app.make_response("")
        resp.headers["Access-Control-Allow-Origin"] = "*"
        resp.headers["Access-Control-Allow-Headers"] = "Content-Type, ngrok-skip-browser-warning, Authorization"
        resp.headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
        resp.status_code = 200
        return resp

# ── Routes ────────────────────────────────────────────────────────────────────
@colab_app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "colab_connected": True, "mode": "colab"})


@colab_app.route("/upload", methods=["POST"])
def upload():
    if "image" not in request.files:
        return jsonify({"error": "No image provided"}), 400

    file = request.files["image"]
    img = Image.open(file).convert("RGB").resize((512, 512))

    save_path = os.path.join(UPLOAD_FOLDER, "room_original.jpg")
    img.save(save_path)

    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=95)

    UPLOAD_STORE["image_b64"] = base64.b64encode(buf.getvalue()).decode()

    return jsonify({"message": "Image uploaded successfully"})


def _get_upload():
    if UPLOAD_STORE["image_b64"]:
        return UPLOAD_STORE["image_b64"]

    p = os.path.join(UPLOAD_FOLDER, "room_original.jpg")

    if os.path.exists(p):
        with open(p, "rb") as f:
            return base64.b64encode(f.read()).decode()

    return None


def _safe(fn, *args, **kwargs):
    """Run fn and always return a JSON-serializable dict, never let an exception
    escape as a raw 500 HTML page (which broke the frontend's error handling)."""
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}


@colab_app.route("/generate", methods=["POST"])
def generate():
    data = request.json or {}

    style = data.get("style")
    palette = data.get("palette")
    custom_prompt = data.get("customPrompt")

    if not style and not custom_prompt:
        return jsonify({"error": "style or customPrompt required"}), 400

    image_b64 = _get_upload()

    if not image_b64:
        return jsonify({"error": "No uploaded image found"}), 400

    result = _safe(generate_style, image_b64, style, palette, custom_prompt)

    if "error" in result:
        return jsonify(result), 500

    return jsonify({
        "message": "Style transfer complete",
        "style": style,
        "image": result["image"]
    })


@colab_app.route("/detect-objects", methods=["POST"])
def detect_objects_route():
    result = _safe(detect_objects, _get_upload() or "")

    if "error" in result:
        return jsonify(result), 500

    return jsonify({
        "message": "Detection complete",
        "objects": result.get("objects", [])
    })


@colab_app.route("/edit-object", methods=["POST"])
def edit_object_route():
    data = request.json or {}

    object_label = data.get("object")
    edit_prompt = data.get("prompt")

    if not object_label or not edit_prompt:
        return jsonify({
            "error": "object and prompt required"
        }), 400

    result = _safe(edit_object, _get_upload() or "", object_label, edit_prompt)

    if "error" in result:
        return jsonify(result), 400

    return jsonify({
        "message": "Edit complete",
        "object": object_label,
        "image": result["image"]
    })


@colab_app.route("/preview-styles", methods=["POST"])
def preview_styles_route():
    data = request.json or {}

    image_b64 = _get_upload()

    if not image_b64:
        return jsonify({
            "error": "No uploaded image found"
        }), 400

    result = _safe(generate_all_previews, image_b64, data.get("palette"))

    if "error" in result:
        return jsonify(result), 500

    return jsonify({
        "message": "Previews generated",
        "previews": result.get("previews", {})
    })


@colab_app.route("/colab-generate", methods=["POST"])
def colab_generate():
    data = request.json or {}
    return jsonify(_safe(
        generate_style,
        data.get("image",""),
        data.get("style"),
        data.get("palette"),
        data.get("customPrompt")
    ))


@colab_app.route("/colab-detect", methods=["POST"])
def colab_detect():
    return jsonify(_safe(detect_objects, request.json.get("image","")))


@colab_app.route("/colab-edit", methods=["POST"])
def colab_edit():
    data = request.json or {}
    return jsonify(_safe(
        edit_object,
        data.get("image",""),
        data.get("object",""),
        data.get("prompt","")
    ))


# ───────── Furnish Room ─────────
@colab_app.route("/colab-furnish", methods=["POST"])
def colab_furnish():
    data = request.json or {}

    image_b64 = data.get("image", "")
    prompt = data.get("prompt", "")

    if not image_b64 or not prompt:
        return jsonify({
            "error": "image and prompt required"
        }), 400

    result = _safe(furnish_room_inpaint, image_b64, prompt)

    if "error" in result:
        return jsonify(result), 500

    return jsonify(result)


@colab_app.route("/colab-preview", methods=["POST"])
def colab_preview():
    data = request.json or {}
    return jsonify(_safe(generate_all_previews, data.get("image",""), data.get("palette")))


# ───────── NEW: Add a specific object from a reference photo (IP-Adapter) ─────────
@colab_app.route("/colab-add-object", methods=["POST"])
def colab_add_object():
    data = request.json or {}

    room_image = data.get("room_image", "")
    object_image = data.get("object_image", "")
    prompt = data.get("prompt", "")

    if not room_image or not object_image:
        return jsonify({"error": "room_image and object_image required"}), 400

    result = _safe(add_object_from_reference, room_image, object_image, prompt)

    if "error" in result:
        return jsonify(result), 500

    return jsonify(result)


# ── Start ─────────────────────────────────────────────────────────────────────

# Serialize model requests to avoid simultaneous GPU inference.
for endpoint, view in list(colab_app.view_functions.items()):
    if endpoint in {"health", "static", "upload"}: continue
    def locked_view(*args, _view=view, **kwargs):
        with MODEL_LOCK:
            return _view(*args, **kwargs)
    colab_app.view_functions[endpoint] = locked_view

## ۹) اجرا و اتصال — این سلول را آخر اجرا کن
your ngrok authtoken را وقتی خواست وارد کن. بعد از چند ثانیه BACKEND_URL و CONNECTION_KEY چاپ می‌شود.

In [ ]:
from getpass import getpass
from pyngrok import ngrok
from werkzeug.serving import make_server
import logging, threading
logging.getLogger("pyngrok").setLevel(logging.CRITICAL)
ngrok.set_auth_token(getpass("Your ngrok authtoken (hidden): "))
server=make_server("127.0.0.1",7860,colab_app,threaded=True)
threading.Thread(target=server.serve_forever,daemon=True).start()
try:
    tunnel=ngrok.connect(7860)
except Exception:
    raise RuntimeError("Tunnel failed. Check your own ngrok authtoken and account.") from None
print("BACKEND_URL:",tunnel.public_url)
print("CONNECTION_KEY:",CONNECTION_KEY)
print("Keep this runtime alive. Connect through the local Flask bridge, not directly from the original frontend.")


## 🎉 تمام شد

سرور بالاست. خروجی سلول قبل را ببین:

- خط `BACKEND_URL` → آدرس ngrok
- خط `CONNECTION_KEY` → کلید اتصال

هر دو را در فرانت (`http://localhost:3000`) داخل پنجرهٔ **Connect AI Backend → Google Colab** وارد کن و Connect بزن.

**نکته:** این سلول تا وقتی که تب Colab را نبندی و رانتایم قطع نشود، به کار خودش ادامه می‌دهد و به هر درخواستی از بک‌اند محلی‌ات جواب می‌دهد. برای توقف، از منو Runtime → Disconnect and delete runtime را بزن.
